# 64장 · LangGraph interrupt와 승인 후 재개

승인이 필요한 그래프는 외부 작업 직전에 실행을 멈추고 상태를 저장합니다. 승인 또는 거절을 전달하면 그 실행을 이어갑니다.

과제 등록 오류 문의를 접수하기 직전에 LangGraph의 interrupt로 실행을 멈춥니다. 같은 실행을 승인 또는 거절로 재개한 뒤 모의 쓰기 도구의 호출 횟수를 비교합니다.

실행을 확인한 커널은 Python 3.12.15입니다. 예제에 사용한 외부 패키지는 langgraph 1.2.16, langgraph-checkpoint 4.3.0입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.


## interrupt()로 승인 대기

아래 그래프의 노드는 interrupt에서 실행을 멈춥니다. 체크포인터가 대기 중인 상태를 보관합니다. 여기서는 수업용 메모리를 쓰므로 커널을 다시 시작하면 기록이 사라집니다.


### 셀 01 · 승인 노드와 체크포인터 구성

노드에서 부수 효과가 생기는 줄은 interrupt 다음에만 둡니다. 같은 승인 번호로 두 번 저장하지 않도록 메모리 사전에서 확인합니다.

처음 interrupt에 닿으면 제안만 보이고 문의 등록은 0건입니다. 같은 작업을 재개하면 이 줄이 승인 또는 거절 값을 반환합니다. 승인 경로만 등록 줄을 지나므로 거절 뒤에는 등록 건수가 0으로 남습니다.


In [1]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt, Command

class Request(TypedDict, total=False):
    approval_id: str
    text: str
    status: str

inquiries = {}
def approve(state: Request):
    decision = interrupt({"approval_id": state["approval_id"], "inquiry_text": state["text"]})
    if decision != "approve":
        return {"status": "declined"}
    inquiries.setdefault(state["approval_id"], state["text"])
    return {"status": "completed"}

builder = StateGraph(Request)
builder.add_node("confirm", approve)
builder.add_edge(START, "confirm")
builder.add_edge("confirm", END)
graph = builder.compile(checkpointer=InMemorySaver())

실행 순서를 연결했습니다. 아직 invoke하지 않았으므로 대기 중인 요청도, 등록된 문의도 없습니다.


### 문의를 보여 주는 시점과 등록하는 시점

<p>문의 내용은 먼저 상태에 담깁니다. inquiries에 등록되는 줄은 interrupt 뒤에 있습니다. 학생이 문의 내용을 볼 수 있어도 아직 등록 건수는 0일 수 있습니다.</p>


### 셀 02 · interrupt()의 대기 상태

대기 응답이 보여 주는 본문과 실제로 기록된 문의 건수를 확인하세요.


In [2]:
config = {"configurable": {"thread_id": "chat-a-demo"}}
pending = graph.invoke({"approval_id": "A1", "text": "과제 등록 오류를 확인해 주세요."}, config)
print(pending["__interrupt__"][0].value["inquiry_text"])
print(len(inquiries))

과제 등록 오류를 확인해 주세요.
0


문의 본문 다음에 0이 나옵니다. 승인 화면에 문구를 띄울 수 있지만 등록 함수는 아직 호출되지 않았습니다.

요청 내용이 화면에 나타난 시점의 실행 횟수는 0입니다. 승인 대기 중에 이미 쓰기 작업을 해 버리면 이후의 거절은 효력이 없습니다. 승인으로 재개한 경로와 거절한 경로에서 실제 실행 횟수를 각각 비교해야 합니다.


### 사람에게 보이는 승인 대상

<p>interrupt는 approval_id와 inquiry_text를 담아 호출자에게 돌려줍니다. 호출자는 __interrupt__의 value에서 문구를 읽고 이 문의를 등록할지 판단합니다.</p>


### confirm에서 멈춘 실행

<p>그래프에는 START → confirm → END가 등록되어 있지만 첫 invoke는 confirm 안에서 중단됩니다. InMemorySaver가 대기 상태를 보관하고, chat-a-demo는 그 상태를 다시 찾을 때 쓰는 thread_id입니다.</p>

<h3>A1 승인과 A2 거절의 결과 읽기</h3>

<p>chat-a-demo에서 A1을 보낸 첫 invoke는 문의 문구를 돌려주고 멈춥니다. 등록은 0건입니다. 같은 config에 Command(resume="approve")를 전달하면 confirm 노드가 재실행되고 A1이 저장됩니다. 별도의 chat-decline에서 A2를 거절하면 declined가 반환되며 A2는 inquiries에 없습니다.</p>


## Command(resume=…)와 체크포인트

재개하려면 같은 thread_id로 체크포인트를 찾아야 합니다. 서비스에서는 여기에 소유자, 만료 시각, 승인 당시 인자의 해시도 대조합니다. 여기서는 로컬 그래프의 멈춤과 재개에 집중합니다.

<!-- lesson-visual practice-space -->
**승인 여부를 바꿔 같은 문의의 등록 건수 비교**

각 선택은 비어 있는 문의 저장소로 시작하는 독립 실행입니다. 같은 체크포인트의 A1에 전달할 결정을 바꿔 보세요.

대기와 거절은 등록 0건입니다. 승인으로 재개한 실행에서만 completed가 되고 문의가 1건 생깁니다.

[웹 교재에서 직접 조작하기](https://langchain-lecture.vercel.app/chapter64/#practice-space)
<!-- /lesson-visual practice-space -->


### 같은 confirm을 다시 실행한 뒤 A1 등록

같은 thread_id로 Command를 보내면 confirm 안의 interrupt가 결정을 반환합니다. 등록은 그 뒤에 한 번 일어납니다.

1. 승인 대기 · thread_id = chat-a-demo
2. confirm 재실행 · 재개 입력 = Command(resume="approve")
3. interrupt 반환 · decision = approve
4. 문의 등록 · inquiries의 새 키 = A1
5. END · status · 총 건수 = completed · 1


### 셀 03 · Command(resume=…)로 승인 전달

기존 대기 요청에 Command를 보냅니다. 같은 질문을 새로 시작하지 않습니다.


In [3]:
done = graph.invoke(Command(resume="approve"), config)
print(done["status"], len(inquiries))

completed 1


completed는 confirm 노드가 반환한 상태이고 1은 inquiries의 항목 수입니다. 대기할 때의 0과 비교하면 승인 결정 뒤에 A1 한 건을 저장했음을 볼 수 있습니다.


#### 승인 뒤에 다시 실행되는 코드

<mark>같은 thread_id로 재개하면 confirm 노드가 처음부터 다시 실행됩니다.</mark> 처음에는 interrupt에서 기다리므로 문의가 저장되지 않습니다. 재개하면 승인값을 읽고 저장하거나 거절로 끝납니다. 저장을 interrupt 앞으로 옮기면 승인 전에 쓰기가 일어납니다.

선수 개념 · 35장 메시지 상태와 대화별 체크포인트

#### 같은 스레드에서 승인 대기를 재개하는 경로

```mermaid
flowchart TD
s["invoke"] --> c["confirm 진입"] --> i{"interrupt의 결정값?"}
i -->|첫 진입| w["체크포인트 · 승인 대기"]
w -->|같은 thread_id로 resume| c
i -->|approve| y["문의 저장 · completed"] --> e["END"]
i -->|그 외| n["저장 생략 · declined"] --> e
```

첫 진입은 interrupt에서 멈춥니다. Command(resume=…)를 같은 스레드에 전달하면 confirm이 처음부터 다시 실행되고, interrupt가 반환한 결정값으로 저장 여부를 정합니다.

### 같은 스레드에서 결정을 전달하기

<p>Command(resume="approve")를 처음과 같은 config로 전달합니다. 중단됐던 confirm 노드는 처음부터 다시 실행되고 interrupt 자리는 이번에 받은 결정을 돌려줍니다. 등록 코드를 그 뒤에 두면 대기 시점에 부작용이 생기지 않습니다.</p>


### 같은 승인 ID가 가리키는 항목

<p>inquiries.setdefault는 해당 ID가 없을 때만 새 값을 넣습니다. 이 예제의 중복 방지는 현재 커널의 딕셔너리 안에서 같은 키를 다시 쓸 때의 동작입니다.</p>


### 과제 04 · 거절 분기의 종료 상태

별도의 대기 요청 A2를 만들고 decline을 보내세요. 기록 목록에 A2가 남는지 확인하세요.


In [4]:
# 새 thread_id로 대기를 만들고 거절한 뒤 건수를 확인하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>other = {&#x27;configurable&#x27;: {&#x27;thread_id&#x27;: &#x27;chat-decline&#x27;}}
graph.invoke({&#x27;approval_id&#x27;: &#x27;A2&#x27;, &#x27;text&#x27;: &#x27;추가 문의&#x27;}, other)
refused = graph.invoke(Command(resume=&#x27;decline&#x27;), other)
print(refused[&#x27;status&#x27;], &#x27;A2&#x27; in inquiries)</code></pre>
<p>실행 결과</p><pre><code>declined False</code></pre>
<p>declined는 거절 분기의 반환값입니다. False는 A2라는 키가 inquiries에 생기지 않았다는 뜻이고 앞서 등록한 A1은 그대로 남습니다.</p>
</details>


### 승인과 거절에서 달라지는 쓰기 횟수

<p>approve 이외의 결정을 받으면 함수는 declined를 바로 반환합니다. 새 문의를 거절했을 때 건수는 거절 직전과 같습니다. 승인 실험을 먼저 했다면 기존 한 건까지 사라지는 것은 아닙니다.</p>


## 복습과 실습 점검

### 과제 05 · 승인한 본문이 처음 제안과 같은지 확인하기

A1로 승인 화면에 보였던 본문과 inquiries에 등록된 값을 비교하세요. 상태가 completed여도 승인한 문구가 그대로 저장됐는지는 본문을 읽어야 알 수 있습니다.


In [ ]:
# pending의 승인 ID로 inquiries를 읽고 제안의 inquiry_text와 비교하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>proposal = pending[&quot;__interrupt__&quot;][0].value
saved_text = inquiries[proposal[&quot;approval_id&quot;]]
print(saved_text == proposal[&quot;inquiry_text&quot;])
print(saved_text)</code></pre>
<p>실행 결과</p><pre><code>True
과제 등록 오류를 확인해 주세요.</code></pre>
<p>두 값이 일치한 범위는 이 커널에서 처리한 A1 한 건입니다. 같은 ID로 다른 내용을 제안하는 경우 setdefault는 원래 값을 보존하므로, 서비스에서는 승인할 당시의 내용과 재개 요청을 함께 확인해야 합니다.</p>
</details>


## 재개하면 어느 노드부터 다시 실행될까요?

앞의 confirm은 승인을 받은 뒤 문의를 저장했습니다. 대기 중인 그래프를 재개하면 confirm에서 interrupt 앞 코드도 다시 실행됩니다. 화면에 보낼 제안 작성과 실제 접수를 노드 두 개로 나누어 이 차이를 확인합니다.

### 셀 06 · 승인 노드의 진입 횟수 관찰하기

방문 목록은 어느 함수에 들어왔는지 관찰하기 위한 수업용 기록입니다. 새 그래프를 준비한 뒤 같은 thread_id에서 대기와 승인을 차례로 실행하세요.

In [5]:
def make_desk_graph(on_accept):
    visits = []
    def confirm_request(state: Request):
        visits.append("confirm")
        decision = interrupt({"id": state["approval_id"], "draft": state["text"]})
        return {"status": "approved" if decision == "approve" else "declined"}
    def save_request(state: Request):
        visits.append("save")
        return {"status": on_accept(state["approval_id"], state["text"])}
    builder = StateGraph(Request)
    builder.add_node("confirm", confirm_request)
    builder.add_node("save", save_request)
    builder.add_edge(START, "confirm")
    builder.add_conditional_edges("confirm", lambda s: s["status"], {"approved": "save", "declined": END})
    builder.add_edge("save", END)
    return builder.compile(checkpointer=InMemorySaver()), visits

desk_graph, visits = make_desk_graph(lambda key, text: "accepted")
thread = {"configurable": {"thread_id": "desk-example"}}
desk_graph.invoke({"approval_id": "desk-2", "text": "파일 등록 문의"}, thread)
print(visits)
desk_graph.invoke(Command(resume="approve"), thread)
print(visits)

['confirm']
['confirm', 'confirm', 'save']


대기 때는 confirm 한 번, 재개 뒤에는 confirm 두 번과 save 한 번이 남습니다. confirm에 실제 문의 등록 코드를 먼저 두면 승인 전에 등록하거나 재개 때 중복 실행할 수 있습니다. 이 예제에서는 승인이 확정된 뒤의 간선만 save로 연결했습니다.

### 과제 07 · 같은 접수 번호가 다시 승인될 때 중복 등록 막기

다른 스레드에서 같은 접수 번호와 본문이 다시 들어오는 경우입니다. on_accept에 넘길 함수를 작성하세요. 새 번호는 등록하고 accepted, 이미 등록한 동일 본문은 already_saved, 같은 번호에 다른 본문은 conflict를 반환하세요. 다른 본문으로 기존 값을 덮어쓰지 마세요.

In [6]:
desk_records = {}
# def save_once(key, text): ...
# 아래 풀이처럼 서로 다른 스레드로 같은 요청을 두 번 승인하고 결과를 확인하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>desk_records = {}
def save_once(key, text):
    if key in desk_records:
        return &quot;already_saved&quot; if desk_records[key] == text else &quot;conflict&quot;
    desk_records[key] = text
    return &quot;accepted&quot;

safe_graph, visits = make_desk_graph(save_once)
def submit_once(thread_id, text):
    cfg = {&quot;configurable&quot;: {&quot;thread_id&quot;: thread_id}}
    safe_graph.invoke({&quot;approval_id&quot;: &quot;desk-3&quot;, &quot;text&quot;: text}, cfg)
    return safe_graph.invoke(Command(resume=&quot;approve&quot;), cfg)[&quot;status&quot;]

print(submit_once(&quot;try-1&quot;, &quot;파일 등록 문의&quot;))
print(submit_once(&quot;try-2&quot;, &quot;파일 등록 문의&quot;))
print(submit_once(&quot;try-3&quot;, &quot;다른 본문&quot;))
print(desk_records)</code></pre>
<p>실행 결과</p><pre><code>accepted
already_saved
conflict
{&#x27;desk-3&#x27;: &#x27;파일 등록 문의&#x27;}</code></pre>
<p>세 번 승인했지만 desk-3 문서는 한 건입니다. 이번 과제의 딕셔너리는 한 커널에서 순서대로 실행한 경우를 확인합니다. 여러 서버가 동시에 같은 번호를 받을 수 있는 서비스에서는 54장에서 본 데이터베이스의 고유 키와 트랜잭션으로도 이 조건을 지켜야 합니다.</p>
</details>

<mark>승인 결과와 실제 저장 여부를 따로 확인하세요.</mark> 같은 승인 번호를 다시 처리할 때는 승인 상태가 같더라도 이미 저장한 문의가 있을 수 있습니다. 저장 함수가 기존 내용까지 확인하면 중복 요청과 충돌을 구별할 수 있습니다.

팀 서비스의 접수 도구를 점검할 때는 재개 시점과 중복 요청을 서로 다른 사례로 검증합니다.